# 02 · Prompting progresivo: de un prompt suelto a conocimiento integrado

**Caso:** clasificar tickets de soporte de telecomunicaciones en `billing`, `technical`, `cancellation` u `other`. El formato de salida siempre será JSON para poder evaluar.

La ruta: empezamos con un prompt sin estructura, vemos por qué falla, le damos estructura y después aplicamos técnicas encima de esa base.

## 0. Preparación

Misma configuración que el notebook 01: llaves en `.env`, LM Studio activo.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

Project root: /Users/andrei/Documents/Projects/Curso LLM Codes/session_2_architecture


Definimos el caso y dos helpers: `call_model` hace la llamada al modelo local y `show` presenta resultados legibles.

In [2]:
import requests
from IPython.display import Markdown, display

LOCAL_URL = f"{settings['lmstudio_base_url'].rstrip('/')}/chat/completions"
LOCAL_MODEL = settings["default_lmstudio_model"]

TICKET = "Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja."
CATEGORIES = ["billing", "technical", "cancellation", "other"]

def call_model(prompt, temperature=0.0, max_tokens=600):
    payload = {
        "model": LOCAL_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = requests.post(LOCAL_URL, json=payload, timeout=120)
    response.raise_for_status()
    message = response.json()["choices"][0]["message"]
    return message.get("content") or message.get("reasoning_content") or ""

def show(title, body):
    display(Markdown(f"#### {title}\n\n{body}\n\n---"))

## 1. Prompt sin estructura

Primero lo que haría cualquiera: pedirle al modelo que clasifique, sin más. Observen la salida: formato libre, categorías inventadas, imposible de evaluar en automático.

> 👉 **Tu turno:** escribe un prompt de una sola línea que pida clasificar `TICKET` y muestra el resultado.

In [3]:
naive_prompt = f"What kind of problem is this? {TICKET}"
show("Prompt", naive_prompt)
show("Result", call_model(naive_prompt))

#### Prompt

What kind of problem is this? Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja.

---

#### Result

Este es un **problema de conectividad o fallo en la línea física** (Line of Sight - LOS) que está afectando tu servicio de internet.

El hecho de que la luz **LOS (Loss of Signal)** siga roja después de reiniciar el módem indica que el equipo no está recibiendo o enviando correctamente la señal desde la central de tu proveedor de servicios (ISP).

Aquí te explico qué significa y qué puedes hacer

---

## 2. El mismo prompt, con estructura

Cinco piezas: rol, categorías cerradas, regla de ambigüedad, entrada delimitada y contrato de salida en JSON. La tarea no cambió; cambió que ahora la salida es predecible y evaluable.

> 👉 **Tu turno:** construye `structured_prompt` con las cinco piezas y compara el resultado con el anterior.

In [4]:
structured_prompt = f"""You classify telecom support tickets.
Choose exactly one category from: {CATEGORIES}.
If the request is ambiguous, choose other.

<ticket>{TICKET}</ticket>

Return only JSON: {{"category": "...", "confidence": 0.0, "reason": "one short sentence"}}
"""
show("Prompt", structured_prompt)
show("Result", call_model(structured_prompt))

#### Prompt

You classify telecom support tickets.
Choose exactly one category from: ['billing', 'technical', 'cancellation', 'other'].
If the request is ambiguous, choose other.

<ticket>Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja.</ticket>

Return only JSON: {"category": "...", "confidence": 0.0, "reason": "one short sentence"}


---

#### Result

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting steps for the modem."
}
```

---

## 3. Estabilidad: ambos prompts a distintas temperaturas

Corremos las dos versiones a tres temperaturas. El prompt sin estructura se degrada al subir la temperatura; el estructurado mantiene el formato incluso con alta diversidad. La estructura es lo que hace al sistema robusto, no la temperatura baja.

> 👉 **Tu turno:** recorre ambos prompts con temperaturas `0.0`, `0.7` y `1.2` y muestra cada resultado.

In [5]:
prompts = {"naive": naive_prompt, "structured": structured_prompt}

for name, prompt in prompts.items():
    for temperature in (0.0, 0.7, 1.2):
        result = call_model(prompt, temperature=temperature)
        show(f"{name} · temperature {temperature}", result)

#### naive · temperature 0.0

Este es un **problema de conectividad o fallo en la línea física** (Loss of Signal).

El hecho de que la luz **LOS (Loss of Signal)** siga roja indica que el módem no está logrando establecer una conexión estable con la red de tu proveedor de servicios de internet (ISP).

Aquí te explico qué significa y qué puedes hacer para intentar solucionarlo:

### 💡 ¿Qué significa la luz LOS roja?

La luz

---

#### naive · temperature 0.7

Este es un problema de **conectividad o servicio** de internet. Específicamente, indica que tu módem no está logrando establecer una conexión física con la red del proveedor de servicios (ISP).

Aquí te explico qué significa y cuáles son los pasos a seguir:

### 1. ¿Qué significa el error?

El código **LOS**

---

#### naive · temperature 1.2

Este es un **problema técnico de conectividad (Internet/Servicio)**.

Específicamente, el mensaje indica

---

#### structured · temperature 0.0

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting steps for the modem."
}
```

---

#### structured · temperature 0.7

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and troubleshooting steps related to modem hardware."
}
```

---

#### structured · temperature 1.2

```json
{
  "category": "technical",
  "confidence": 1.0,
  "reason": "The ticket describes a loss of signal and modem errors, which is a technical issue."
}
```

---

## 4. One-shot

Sobre la base estructurada añadimos un ejemplo representativo. El ejemplo enseña a la vez la decisión y el formato; conviene que sea correcto, breve y cercano al borde que queremos aclarar.

> 👉 **Tu turno:** extiende el prompt estructurado con un ejemplo etiquetado de `billing`.

In [ ]:
one_shot_prompt = ""
...

## 5. Few-shot

Varios ejemplos cubren clases y casos límite. No se trata de llenar contexto: cada ejemplo debe aportar una regla observable. Ojo con el orden y con sobrerrepresentar una clase.

> 👉 **Tu turno:** arma cuatro ejemplos balanceados, uno por categoría, y clasifica `TICKET`.

In [ ]:
examples = []
few_shot_prompt = ""
...

## 6. Chain-of-Thought (CoT)

La técnica clásica pide razonar paso a paso antes de responder. En producción suele ser mejor pedir una **justificación breve y verificable** con criterios observables: síntoma, evidencia y decisión.

> 👉 **Tu turno:** escribe un prompt con tres criterios observables en orden y pide justificación basada en evidencia.

In [ ]:
cot_prompt = ""
...

## 7. Knowledge Generation & Knowledge Integration

Dos llamadas: primero generamos conocimiento del dominio, después lo integramos como contexto controlado para decidir. Esto permite inspeccionar, filtrar o sustituir el conocimiento antes de clasificar.

**Antecesor:** *Generated Knowledge Prompting* generaba conocimiento y lo añadía al prompt final. Separar las dos etapas mejora el control.

> 👉 **Tu turno:** pide reglas del dominio sin permitir la clasificación final.

In [ ]:
knowledge_prompt = ""
generated_knowledge = ""
...

Segunda etapa: inyectamos el conocimiento revisado y pedimos evidencia más un campo `used_knowledge` para auditar qué se usó.

> 👉 **Tu turno:** inyecta `generated_knowledge` en un segundo prompt y clasifica.

In [ ]:
integration_prompt = ""
...

## 8. Salida estructurada garantizada (Pydantic + JSON Schema)

Hasta ahora el JSON se pedía «por favor» dentro del prompt: el modelo puede romper el contrato en cualquier momento. La forma robusta es declararlo con un modelo de Pydantic y pasarlo como `response_json_schema`: el proveedor fuerza la salida a cumplir el esquema y la respuesta se valida de regreso a un objeto tipado. Lo aplicamos sobre nuestra mejor técnica: el prompt few-shot.

> 👉 **Tu turno:** define los campos `category` (Literal con las cuatro categorías), `confidence` y `reason` en `TicketClassification`; luego haz la llamada con `response_json_schema` y valida con `model_validate_json`.

In [ ]:
from typing import Literal
from google import genai
from pydantic import BaseModel, Field

client = genai.Client(api_key=settings["google_api_key"])

class TicketClassification(BaseModel):
    ...

In [ ]:
...

## 9. Cierre

- Zero-shot estructurado es la línea base; suele bastar para tareas claras.
- One/few-shot ayudan cuando la frontera entre clases necesita ejemplos.
- CoT o criterios estructurados sirven cuando la decisión requiere composición.
- Knowledge Generation & Integration sirve cuando hace falta conocimiento inspeccionable.
- Una técnica más compleja solo se conserva si mejora una métrica relevante.
- Cuando el formato es crítico no se pide: se garantiza con `response_json_schema`.

<details><summary><strong>Pausa y conclusión</strong></summary>

1. ¿Qué cambió entre el prompt suelto y el estructurado?
2. ¿Cuál de los dos resistió mejor la temperatura alta y por qué?
3. ¿Cuándo justificarían pagar dos llamadas (knowledge + integration)?
</details>